# 🚘 CrashLens — Vehicle Part Segmentation
## Stage 2 / exp03 — CrashCar101 + cropped carparts-seg, targeting PART CLASSIFICATION

Reference: `docs/segmentation_finetuning.md` (§5 Stage 2, §7 logging convention, §8 notes
template), `exp02_finetune_cropped_carparts.ipynb`, `segmentation_baseline_eval.ipynb` (exp01).

**Reference to beat — exp02_finetune_cropped_carparts (Stage 2, fine-tuned on
cropped carparts-seg data, same pretrained checkpoint as exp01):**
- Real close-ups (`cropped/`): **55.0%** detection rate (11/20), up from exp01's 10.0%.
- Real full-car (`full/`): **100.0%** detection rate (20/20), up from exp01's 95.0%.
- **Known weakness (from reviewing exp02's `val_preds/cropped/` overlays): PART
  CLASSIFICATION is poor even when something is detected** — e.g. shattered front
  glass predicted as `hood`. exp02's `cropped/` instance distribution is dominated
  by `hood` (6 of 13 instances) relative to more plausible classes, which is
  consistent with this. exp02 never measured this directly — no per-part accuracy
  number exists yet, only detection rate + raw class counts.

**Goal of this experiment.** Test whether adding **CrashCar101** (synthetic,
procedurally-damaged 3D car renders with pixel-accurate part + damage segmentation
masks — Parslov et al., WACV 2024, `JensParslov/CrashCar` on Hugging Face) to the
cropped-carparts training data improves **part classification** on our real
close-ups — not just detection rate. Primary metric: **per-part accuracy** (does the
model's top prediction on a real close-up match the actual damaged part?), plus a
confusion matrix. Detection rate is still reported for exp01/exp02 comparability.

**⚠️ TWO variables change vs exp02, not one:** (1) training data now includes
CrashCar101 in addition to cropped carparts, and (2) the training recipe adds
overfitting-prevention (early stopping, weight decay, a backbone-freeze warmup phase)
that exp02 did not use. This is intentional per your instructions, but it means a
gain (or regression) can't be cleanly attributed to the dataset change alone. **notes.md
(Section 15) recommends an ablation run — dataset-only, exp02's original recipe — if
attribution turns out to matter.**

**This notebook is NOT executed here — scaffolded only, per your instruction. You run
it in Colab.**

**Assumptions / things that need your attention before running — read all of these:**

1. **Drive root** — reuses exp01/exp02's actual root, `MyDrive/carparts/segmentation/...`
   (not the doc's literal example `Drive/CrashLens/segmentation/...`), for continuity.
2. **CrashCar101's class taxonomy — resolved, sourced from `cgpart_3d.zip` + CrashCar101's
   own source code (thank you for pointing at the Downloads copy).** The paper (§3) doesn't
   list classes inline, but states it "follow[s] a part taxonomy from [33]" (CGPart, Liu et
   al.) and derives its 27 car part classes by merging CGPart's 4 wheel parts into 1 and its
   2 license-plate parts into 1. I confirmed this two ways:
   - `cgpart_3d.zip` → `CGPart/labels/car/*.json` (5 car models) — each maps a part name to a
     list of mesh vertex/face indices. Union across all 5 = 31 names, exactly the "31-category
     taxonomy" the paper says it starts from: 4 wheel parts (`front/back_left/right_wheel`),
     2 plate parts (`front/back_license_plate`), plus 25 others.
   - CrashCar101's own generation code (`scene_class.py` in
     `github.com/JensPars/CrashCar_procedural_generation`) hardcodes the identical CGPart names
     in `self.bump_keys` / `self.window_keys` / `self.light_keys`, and merges wheels via
     `self.wheel_names = self.parts["wheel"]` — confirming CrashCar101 uses CGPart's names
     verbatim (not a renamed/different scheme), consolidated per the paper's rule.
   - 11 (bump/frame/roof/hood/trunk) + 8 (windows) + 4 (lights) + 1 (wheel, merged) +
     2 (mirrors, NOT merged — paper only mentions merging wheels/plates) + 1 (plate, merged)
     = **27**, matching the paper's stated count exactly.

   `CRASHCAR_TO_CANONICAL` (Section 4b) is filled in with this 27-class mapping. **Two things
   still need runtime confirmation, not taxonomy guessing:**
   - The *numeric pixel-id ↔ name* binding in the actual distributed masks — `pass_index` is
     assigned by `enumerate(self.parts)` in the generator, and `self.parts`'s key order is
     loaded per-3D-model from an external file, so the id order isn't fixed/documented. Section
     5b discovers whatever the loaded dataset actually exposes (named masks, indexed pixel
     values, or RGB) and validates against the name list below rather than assuming an order.
   - **Correction (verified 2026-08-02 by directly inspecting the zip contents, not just the
     HF file-listing UI):** `damage.zip` and `parts.zip` are plain zip archives of one PNG mask
     per rendered frame (`<type>/<model-hash>/<frame>.png`, same relative path as `img/`) — not
     pickled dict-of-masks as an earlier pass here assumed. A PNG mask can't carry a `named`
     (dict) representation, so Section 5b will report `indexed`.
   - **Per-scene legend — RESOLVED (2026-08-02), not the 5 cgpart_3d.zip car models:**
     `cgpart_3d.zip` only ships 5 car meshes, but the distributed `img/` folder has ~90+
     distinct scene hashes, and checking overlap found only 1 of them matches a cgpart_3d.zip
     hash — the other 89+ use meshes cgpart_3d.zip doesn't include, and (checked directly) the
     5 known meshes don't even share a consistent part-name ORDER with each other, so no fixed
     global legend exists. The actual source: CrashCar101's generator repo
     (`github.com/JensPars/CrashCar_procedural_generation`) ships `data/accepted/<hash>.txt` —
     one manifest per scene, `{part_name: [mesh_ids]}`, and the hash in the filename matches the
     `img/<hash>/` scene folder exactly. `scene_class.py`'s actual assignment loop is
     `for idx, key in enumerate(self.parts): if len(self.parts[key]) > 0: ob.pass_index = idx + 1`
     — so `{idx+1: key for idx, key in enumerate(manifest.keys())}` is the exact, sourced (not
     guessed) per-scene id->name legend. Section 5a now fetches these manifests (public repo, no
     HF token needed) for every scene actually used and attaches the right legend per sample —
     see Section 5a's `SCENE_LEGENDS` and Section 7's `load_sample`. A scene whose manifest is
     missing/unreachable is dropped before download (logged), not silently mislabeled.
   - **Known gap, not something to "fix" by guessing:** CGPart/CrashCar101 has no class for
     the four fenders at all (checked — no `fender`/`panel`/`sill`/`rocker`-like key in any of
     the 5 CGPart car models). `left_frame`/`right_frame` map cleanly to `left_sill`/
     `right_sill` (matches your original brief), but `front/back_left/right_fender` in
     `CANONICAL_CLASSES` will get **zero training instances from CrashCar101** — they stay
     canonical (still needed for `labor_hours_lookup.py` compatibility) but this experiment
     can't teach the model to recognize them. Flagged again in notes.md.
3. **CrashCar101 is semantic-segmentation masks (pixel value = class id/color), not YOLO
   polygon labels.** Section 7 converts masks → per-instance YOLO-seg polygons via connected
   components + contour extraction. This is a reasonable approach for reasonably-separated
   procedurally-rendered parts, but review a few converted labels visually before trusting it
   at scale — mask-to-polygon can merge adjacent same-class blobs into one "instance."
4. **CrashCar101 is ~101,050 images** — far larger than cropped-carparts (~15.6k train crops
   from exp02) and infeasible to use in full on a single Colab GPU for a "start minimal, one
   variable at a time" run. **Section 5a subsamples the remote file list before downloading
   anything** (`CRASHCAR_SAMPLE_FRAC` / `CRASHCAR_MAX_IMAGES` in Section 3, fixed seed) — moved
   earlier than Section 6 specifically so the *download* is bounded by this cap too, not just
   the converted dataset (the original version downloaded the full ~83,730-file repo regardless
   of this setting). Tune the cap, but keep it fixed for any run you want to compare against.
5. **No ground-truth part labels for the real CrashLens close-ups — deliberately not built.**
   Manually labeling the real images is expensive, so per your direction this notebook uses a
   different design: **quantitative per-part metrics (precision/recall/mAP, confusion matrix)
   are measured on a held-out slice of CrashCar101's OWN labels** (Section 11) — no manual
   labeling needed, since Section 5a computes its own fixed-seed 70/15/15 train/val/test split
   over the raw file listing (the paper's official 83,604/8,311/9,135 split isn't reachable
   through the repo's raw `img/` files, and the `datasets.load_dataset` path that could reach it
   was failing in practice — see Section 5a) and we simply never train or tune on the test
   portion. **Read this caveat before trusting that number:** it measures
   whether the model learned to distinguish parts *within the synthetic domain* — it does
   **not** by itself prove that transfers to real photos, which is the actual question this
   experiment exists to answer. Real-domain assessment stays **manual/qualitative** (Section
   12): rather than blind-scrolling all 40 real overlays, Section 12 uses exp02's own saved
   `real_eval_results.csv` to surface the specific `cropped/` images exp02 mis-predicted as
   `hood` (the pattern you already flagged), and points you at exp03's matching overlay for a
   direct before/after look — you record the verdict in `notes.md` (Section 14), which already
   has free-text Strengths/Weaknesses fields for exactly this.
6. **Real-domain validation set.** `val` is CrashCar101(held-out) + cropped-carparts(val) —
   both synthetic-leaning, not real CrashLens photos — so early stopping/model selection
   optimizes for the synthetic domain, not the real one. Given assumption #5's design (no real
   ground truth at all now, not even for a val set), there's no real-domain signal anywhere in
   this pipeline except the manual Section 12 check at the very end — flagged again in notes.md
   as the main limitation of this experiment's methodology.
7. **Backbone freeze "for the first epochs".** Ultralytics' `freeze=N` argument freezes the
   first N layers for the *entire* training call, not just a warmup window — there's no native
   "unfreeze after epoch K" flag. Section 10 implements the warmup via **two sequential
   `model.train()` calls**: a short frozen-backbone phase, then a longer unfrozen phase
   resuming from the frozen phase's `last.pt`. Flag if you intended something else.
8. **`MODEL_SOURCE`** (Section 3) is the **same pretrained carparts-seg checkpoint exp01 and
   exp02 both started from** (not exp02's fine-tuned weights) — keeps this comparable to exp02
   as "same starting point, different data + regularization," matching exp02's own relationship
   to exp01.
9. Per your constraints: no backend/Flutter files touched, labor-hours table untouched — this
   notebook is fully standalone. Canonical class **names** are chosen to exactly match
   `backend_api/services/labor_hours_lookup.py` (the `LOOKUP` + `WHEEL_HOURS` + `NON_COSTED` +
   `UNMAPPED_PARTS` keys) so no further remap is needed at integration time.
10. **Section 5a download restructured (2026-08-02)** to fix a real resource/time problem: it
    used to call `snapshot_download` with no filter, which pulled the *entire* repo (all
    ~83,730 individual `img/*.png` files, one HTTP request each, plus `damage.zip`) regardless
    of `CRASHCAR_SAMPLE_FRAC`/`CRASHCAR_MAX_IMAGES` — those only trimmed what got *converted*,
    not what got *downloaded*. Section 5a now lists `img/` remotely, subsamples that list, and
    downloads only the selected files in parallel. `damage.zip` is no longer downloaded at all —
    nothing in this notebook consumes damage masks; re-add it if a future experiment needs them.


## 📁 Section 1 — Drive mount & experiment folders

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE_ROOT = '/content/drive/MyDrive/carparts/segmentation'
EXP_ID     = 'exp03_crashcar_plus_carparts'
EXP_DIR    = os.path.join(DRIVE_ROOT, 'experiments', EXP_ID)

for sub in ['weights', 'val_preds/full', 'val_preds/cropped', 'confusion_matrix']:
    os.makedirs(os.path.join(EXP_DIR, sub), exist_ok=True)

print('✅ Drive mounted and experiment folders ready.')
print(f'   Experiment root: {EXP_DIR}')


Mounted at /content/drive
✅ Drive mounted and experiment folders ready.
   Experiment root: /content/drive/MyDrive/carparts/segmentation/experiments/exp03_crashcar_plus_carparts


## 📦 Section 2 — Install dependencies

In [2]:
!pip install -q ultralytics huggingface_hub datasets
import ultralytics
ultralytics.checks()


Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
Setup complete ✅ (12 CPUs, 167.1 GB RAM, 46.7/235.7 GB disk)


## ⚙️ Section 3 — Configuration (edit before running)

In [3]:
import torch
import pandas as pd
import os

# ── IDENTITY WITH exp01/exp02 — same starting checkpoint, same real eval data ──
MODEL_SOURCE   = '/content/drive/MyDrive/carparts/weights/best.pt'  # TODO: confirm same path exp01/exp02 used
REAL_DATA_ROOT = '/content/drive/MyDrive/carparts'                  # contains full/, cropped/ (real CrashLens images) — detection-rate + manual review only, no GT
CONF_THRESH    = 0.25   # identical to exp01/exp02 — keep for comparability
IOU_THRESH     = 0.7    # identical to exp01/exp02
IMG_SIZE       = 320    # Lowered for resource management: was 640
IMG_EXTS       = ('.jpg', '.jpeg', '.png')
SUBFOLDERS     = ['full', 'cropped']

SEED            = 42     # same seed as exp02
DATASET_VERSION = 'v2_crashcar_plus_carparts'
LOCAL_BUILD_ROOT   = '/content/unified_build'
DRIVE_DATASETS_DIR = os.path.join(DRIVE_ROOT, 'datasets', DATASET_VERSION)  # frozen, versioned archive (docs §7)

# exp02's already-frozen cropped-carparts dataset — REUSED, not rebuilt (assumption: exp02 has been run at least once)
EXP02_DATASET_DIR = os.path.join(DRIVE_ROOT, 'datasets', 'v1_cropped_carparts')

# ── CrashCar101 acquisition ─────────────────────────────────────────────────
CRASHCAR_HF_REPO   = 'JensParslov/CrashCar'
# CHANGED: Download raw CrashCar101 data directly to Drive to prevent local disk space issues
CRASHCAR_LOCAL_DIR = os.path.join(DRIVE_ROOT, 'crashcar101_raw_repo') # Modified to download to Drive
# Subsampling (assumption #4) — CrashCar101 is ~101,050 images, far larger than the
# ~15.6k cropped-carparts train crops from exp02; using it in full is both infeasible
# for a single-GPU Colab run and would swamp the real-photo-derived data entirely.
CRASHCAR_SAMPLE_FRAC = 0.05   # Lowered for resource management: was 0.15
CRASHCAR_MAX_IMAGES  = 2000  # Lowered for resource management: was 12000
# TEST split (assumption #5) — held out from training/val entirely, used ONLY for the
# quantitative per-part metrics in Section 11. Bigger than a quick sanity check needs, but
# it's evaluated once, so the extra reliability is cheap relative to training cost.
CRASHCAR_TEST_SAMPLE_FRAC = 0.05 # Lowered for resource management: was 0.20
CRASHCAR_TEST_MAX_IMAGES  = 500  # Lowered for resource management: was 2500

# ── Overfitting-prevention (new vs exp02 — see assumption #7 for the freeze mechanics) ──
FREEZE_LAYERS     = 10     # Ultralytics layer-count to freeze during the warmup phase
FREEZE_EPOCHS     = 10     # warmup phase length (backbone frozen)
FINETUNE_EPOCHS   = 90     # main phase length (backbone unfrozen), resumes from the frozen phase's last.pt
PATIENCE          = 15     # Ultralytics early-stopping patience (epochs with no val mAP improvement), applied in the main phase
WEIGHT_DECAY      = 0.001  # Ultralytics default is 5e-4; doubled given the added synthetic data volume
# Augmentation — Ultralytics built-ins, explicit so it's auditable (values are library defaults
# unless noted). Kept ON: mosaic, hsv jitter, flips, translate/scale. Kept OFF: mixup, copy_paste
# (both blend/duplicate whole instances, which risks producing incoherent part boundaries on
# already-synthetic renders — not something we're deliberately testing here).
AUGMENT = dict(
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,   # color jitter — library defaults
    degrees=0.0, translate=0.1, scale=0.5, shear=0.0,  # geometric — library defaults
    fliplr=0.5, flipud=0.0,              # horizontal flip only (vertical flip is not physically meaningful for a car)
    mosaic=1.0,                          # on — library default
    mixup=0.0, copy_paste=0.0,           # off — see note above
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Config set. Device={DEVICE}')
print(f'   EXP02_DATASET_DIR: {EXP02_DATASET_DIR}')
print(f'   DATASET_VERSION  : {DATASET_VERSION}  (seed={SEED})')

✅ Config set. Device=cuda
   EXP02_DATASET_DIR: /content/drive/MyDrive/carparts/segmentation/datasets/v1_cropped_carparts
   DATASET_VERSION  : v2_crashcar_plus_carparts  (seed=42)


## 🗂️ Section 4 — Canonical class set + explicit source→canonical mapping

One canonical class set for this experiment, chosen to **exactly match**
`backend_api/services/labor_hours_lookup.py`'s keys, so segmentation output needs
**no further remap** at integration time (per your instructions).


In [4]:
# 24 canonical classes = labor_hours_lookup.py's LOOKUP keys + WHEEL_HOURS ('wheel') +
# NON_COSTED (left/right_mirror) + UNMAPPED_PARTS keys (fenders/sill/roof — parts
# carparts-seg has no class for, but CrashCar101 does). Order mirrors that file for
# easy cross-reference; order otherwise doesn't matter to YOLO (ids are positional).
CANONICAL_CLASSES = [
    'front_bumper', 'hood', 'front_glass', 'front_left_light', 'front_right_light',
    'front_left_door', 'front_right_door', 'back_left_door', 'back_right_door',
    'back_bumper', 'back_glass', 'back_left_light', 'back_right_light', 'trunk',
    'wheel', 'left_mirror', 'right_mirror',
    'front_left_fender', 'front_right_fender', 'back_left_fender', 'back_right_fender',
    'roof', 'left_sill', 'right_sill',
]
CANONICAL_INDEX = {name: i for i, name in enumerate(CANONICAL_CLASSES)}
assert len(CANONICAL_CLASSES) == len(set(CANONICAL_CLASSES)) == 24

# ── carparts-seg (23 classes) → canonical ───────────────────────────────────
# Grounded in the ACTUAL class list Ultralytics' carparts-seg.yaml returned in
# exp01/exp02 (see their Section 4/'Section 4' cell outputs) — not guessed.
# `None` = no canonical target -> DROP that instance (per your instructions).
# tailgate -> trunk mirrors labor_hours_lookup.py's ALIASES = {'tailgate': 'trunk'}.
CARPARTS_TO_CANONICAL = {
    'back_bumper':       'back_bumper',
    'back_door':         None,   # ambiguous side (no left/right distinction) -> drop
    'back_glass':        'back_glass',
    'back_left_door':    'back_left_door',
    'back_left_light':   'back_left_light',
    'back_light':        None,   # ambiguous side -> drop
    'back_right_door':   'back_right_door',
    'back_right_light':  'back_right_light',
    'front_bumper':       'front_bumper',
    'front_door':          None,  # ambiguous side -> drop
    'front_glass':         'front_glass',
    'front_left_door':     'front_left_door',
    'front_left_light':    'front_left_light',
    'front_light':          None,  # ambiguous side -> drop
    'front_right_door':    'front_right_door',
    'front_right_light':   'front_right_light',
    'hood':                 'hood',
    'left_mirror':          'left_mirror',
    'object':                None,  # generic/undefined class -> drop
    'right_mirror':          'right_mirror',
    'tailgate':               'trunk',   # alias, matches labor_hours_lookup.py
    'trunk':                  'trunk',
    'wheel':                  'wheel',
}
_dropped = [k for k, v in CARPARTS_TO_CANONICAL.items() if v is None]
_bad = [v for v in CARPARTS_TO_CANONICAL.values() if v is not None and v not in CANONICAL_INDEX]
assert not _bad, f'CARPARTS_TO_CANONICAL targets not in CANONICAL_CLASSES: {_bad}'
print(f'✅ CARPARTS_TO_CANONICAL: {len(CARPARTS_TO_CANONICAL)} source classes, '
      f'{len(CARPARTS_TO_CANONICAL) - len(_dropped)} kept, {len(_dropped)} dropped (ambiguous side / generic): {_dropped}')


✅ CARPARTS_TO_CANONICAL: 23 source classes, 18 kept, 5 dropped (ambiguous side / generic): ['back_door', 'back_light', 'front_door', 'front_light', 'object']


In [5]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
print("HF_TOKEN retrieved: (value hidden)" if HF_TOKEN else "HF_TOKEN not found or empty.")

HF_TOKEN retrieved: (value hidden)


In [6]:
import os
import json
import zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import requests
from huggingface_hub import login, whoami, list_repo_files, hf_hub_download
from google.colab import userdata

os.environ["HF_HUB_DISABLE_XET"] = "1"      # bypass the failing xet token handshake

HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
    raise ValueError("Hugging Face token not found in Colab secrets. Please add it as 'HF_TOKEN'.")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print(whoami()["name"])   # sanity check: should print your username, not error

os.makedirs(CRASHCAR_LOCAL_DIR, exist_ok=True)

# ── RESTRUCTURED (was: snapshot_download of the whole repo, subsample after) ──
# Old behavior downloaded all ~83,730 individual img/*.png files (one HTTP request each)
# plus damage.zip regardless of CRASHCAR_SAMPLE_FRAC/CRASHCAR_MAX_IMAGES — the subsample
# only trimmed what got *converted* in Section 6/7, not what got *downloaded* here. That's
# the actual source of the slow/heavy runs, not the subsample settings themselves. New order:
#   1. download+extract the one small full-archive of part masks (parts.zip, ~342MB)
#   2. list img/*.png remotely (metadata only, no image bytes)
#   3. resolve each scene's id->name legend from the CrashCar101 generator repo's
#      data/accepted/<hash>.txt manifests (see cell 0, assumption #2) and drop any scene
#      we can't legend — this is what actually makes the 'indexed' part masks usable at all
#   4. split + subsample the remaining (legend-resolved, mask-paired) file list
#   5. download ONLY the selected image files, in parallel
# damage.zip is intentionally not downloaded — nothing downstream in this notebook consumes
# damage masks (this experiment targets part classification only); re-add it if a future
# experiment needs it.

def ensure_extracted_zip(zip_name, expected_dirnames):
    """Download+extract a repo-root zip once; a no-op on re-runs once extracted."""
    existing = next((d for d in expected_dirnames if (Path(CRASHCAR_LOCAL_DIR) / d).is_dir()), None)
    if existing:
        print(f'✅ {zip_name} already extracted -> {Path(CRASHCAR_LOCAL_DIR) / existing}')
        return existing
    print(f'▶ Downloading {zip_name}...')
    zip_path = hf_hub_download(repo_id=CRASHCAR_HF_REPO, filename=zip_name, repo_type='dataset',
                                local_dir=CRASHCAR_LOCAL_DIR, token=HF_TOKEN)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(CRASHCAR_LOCAL_DIR)
    found = next((d for d in expected_dirnames if (Path(CRASHCAR_LOCAL_DIR) / d).is_dir()), None)
    if not found:
        print(f'⚠️  Extracted {zip_name} but none of {expected_dirnames} appeared under {CRASHCAR_LOCAL_DIR}.')
        print(f'    Top-level contents now: {sorted(p.name for p in Path(CRASHCAR_LOCAL_DIR).iterdir())}')
        raise RuntimeError(f'{zip_name} did not extract into any of the expected dir names '
                            f'{expected_dirnames} — inspect the printed contents above and adjust '
                            'expected_dirnames (or this notebook\'s part-mask handling) accordingly.')
    print(f'✅ Extracted {zip_name} -> {Path(CRASHCAR_LOCAL_DIR) / found}')
    return found


PART_MASK_DIR = Path(CRASHCAR_LOCAL_DIR) / ensure_extracted_zip('parts.zip', ['part', 'parts'])

# ── list img/*.png remotely (one metadata call, no image bytes) ─────────────
print('▶ Listing img/*.png in the repo (remote, no download)...')
all_img_files = [f for f in list_repo_files(CRASHCAR_HF_REPO, repo_type='dataset', token=HF_TOKEN)
                  if f.startswith('img/') and f.endswith('.png')]
print(f'✅ {len(all_img_files)} raw render files listed.')


def _scene_hash(img_rel_path):
    return Path(img_rel_path).parts[1]  # img/<hash>/<frame>.png


def _part_relpath(img_rel_path):
    return Path(img_rel_path).relative_to('img').as_posix()


# ── resolve each scene's id->name legend (source: CrashCar101's OWN generator repo) ──
# scene_class.py's real assignment loop (verified against the repo directly, not guessed):
#   for idx, key in enumerate(self.parts):
#       if len(self.parts[key]) > 0:
#           ob.pass_index = idx + 1
# self.parts is exactly data/accepted/<scene_hash>.txt, a {part_name: [mesh_ids]} dict, and
# that filename hash matches the img/<hash>/ scene folder hash 1:1. This is public (no HF
# token) and per-SCENE, not per-frame — only ~90-100 tiny fetches total, not one per image.
GITHUB_MANIFEST_URL = 'https://raw.githubusercontent.com/JensPars/CrashCar_procedural_generation/main/data/accepted/{hash}.txt'

unique_scene_hashes = sorted({_scene_hash(f) for f in all_img_files})
print(f'▶ Resolving id->name legends for {len(unique_scene_hashes)} scenes from the generator repo...')

SCENE_LEGENDS = {}
_missing_legend_scenes = []
for h in unique_scene_hashes:
    resp = requests.get(GITHUB_MANIFEST_URL.format(hash=h), timeout=15)
    if resp.status_code != 200:
        _missing_legend_scenes.append(h)
        continue
    manifest = json.loads(resp.text)
    SCENE_LEGENDS[h] = {idx + 1: key for idx, key in enumerate(manifest.keys())}

print(f'✅ Resolved legends for {len(SCENE_LEGENDS)}/{len(unique_scene_hashes)} scenes.')
if _missing_legend_scenes:
    print(f'⚠️  {len(_missing_legend_scenes)} scene(s) have no manifest on the generator repo — '
          f'dropped, not guessed: {_missing_legend_scenes[:10]}{"..." if len(_missing_legend_scenes) > 10 else ""}')
assert SCENE_LEGENDS, 'No scene legends resolved at all — check GITHUB_MANIFEST_URL / network access before continuing.'

# ── keep only entries that have BOTH a matching part mask on disk AND a resolved legend ──
part_mask_relpaths = {p.relative_to(PART_MASK_DIR).as_posix() for p in PART_MASK_DIR.rglob('*.png')}
print(f'   {len(part_mask_relpaths)} part-mask files found under {PART_MASK_DIR}.')

paired_img_files = [f for f in all_img_files
                     if _part_relpath(f) in part_mask_relpaths and _scene_hash(f) in SCENE_LEGENDS]
print(f'✅ {len(paired_img_files)} files have both a part mask and a resolved legend (usable pairs).')
assert paired_img_files, (
    f'No usable img/part/legend triples found. {PART_MASK_DIR} may use a different internal layout '
    f'than img/, or SCENE_LEGENDS resolution failed — inspect the prints above before continuing.'
)

# ── split + subsample the FILE LIST, before downloading any image bytes ─────
# CrashCar101's official train/val/test split isn't reachable through this raw-file listing
# (no split metadata ships alongside img/), and the datasets.load_dataset path that could
# read it has been failing (BuilderConfig/allow_unsafe_code error) — so this always uses the
# same fixed-seed 70/15/15 fallback the old code already fell back to in practice.
def split_refs(paths, seed=SEED):
    rng = np.random.RandomState(seed)
    order = rng.permutation(len(paths))
    n = len(paths)
    cut_train, cut_val = int(0.70 * n), int(0.85 * n)
    return ([paths[i] for i in order[:cut_train]],
            [paths[i] for i in order[cut_train:cut_val]],
            [paths[i] for i in order[cut_val:n]])


def subsample(paths, frac, cap, seed=SEED):
    rng = np.random.RandomState(seed)
    k = min(cap, int(len(paths) * frac))
    chosen = rng.choice(len(paths), size=k, replace=False)
    return sorted(paths[i] for i in chosen)


_train_all, _val_all, _test_all = split_refs(paired_img_files)
cc_train_files = subsample(_train_all, CRASHCAR_SAMPLE_FRAC, CRASHCAR_MAX_IMAGES)
cc_val_files = subsample(_val_all, CRASHCAR_SAMPLE_FRAC, max(1, CRASHCAR_MAX_IMAGES // 5))
cc_test_files = subsample(_test_all, CRASHCAR_TEST_SAMPLE_FRAC, CRASHCAR_TEST_MAX_IMAGES)
selected_files = cc_train_files + cc_val_files + cc_test_files
print(f'✅ Selected: train={len(cc_train_files)}, val={len(cc_val_files)}, test={len(cc_test_files)} '
      f'({len(selected_files)} total, out of {len(paired_img_files)} usable pairs; seed={SEED})')

# ── download ONLY the selected image files, in parallel ─────────────────────
def download_selected(files, max_workers=16):
    def _one(rel_path):
        return hf_hub_download(repo_id=CRASHCAR_HF_REPO, filename=rel_path, repo_type='dataset',
                                local_dir=CRASHCAR_LOCAL_DIR, token=HF_TOKEN)

    done = 0
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(_one, f) for f in files]
        for fut in as_completed(futures):
            fut.result()  # surface any download error immediately
            done += 1
            if done % 1000 == 0 or done == len(files):
                print(f'   downloaded {done}/{len(files)}')


print(f'▶ Downloading {len(selected_files)} selected img/*.png files '
      f'(was {len(all_img_files)} files in the old full-repo download)...')
download_selected(selected_files)
print('✅ Selected images downloaded. (Already-present files were skipped — safe to re-run.)')


def build_samples(img_rel_paths):
    return [{'image': str(Path(CRASHCAR_LOCAL_DIR) / f),
             'part': str(PART_MASK_DIR / _part_relpath(f)),
             'scene': _scene_hash(f)} for f in img_rel_paths]


crashcar_source = 'raw_files'
crashcar_train_data = build_samples(cc_train_files)
crashcar_val_data = build_samples(cc_val_files)
crashcar_test_data = build_samples(cc_test_files)
crashcar_data = crashcar_train_data + crashcar_val_data + crashcar_test_data  # combined view, used only by Section 5b's sampling
print(f'✅ Sample lists ready: train={len(crashcar_train_data)}, val={len(crashcar_val_data)}, '
      f'test={len(crashcar_test_data)}')


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


lele22e
✅ parts.zip already extracted -> /content/drive/MyDrive/carparts/segmentation/crashcar101_raw_repo/parts
▶ Listing img/*.png in the repo (remote, no download)...
✅ 83725 raw render files listed.
▶ Resolving id->name legends for 92 scenes from the generator repo...
✅ Resolved legends for 92/92 scenes.
   83725 part-mask files found under /content/drive/MyDrive/carparts/segmentation/crashcar101_raw_repo/parts.
✅ 83725 files have both a part mask and a resolved legend (usable pairs).
✅ Selected: train=2000, val=400, test=500 (2900 total, out of 83725 usable pairs; seed=42)
▶ Downloading 2900 selected img/*.png files (was 83725 files in the old full-repo download)...
   downloaded 1000/2900
   downloaded 2000/2900
   downloaded 2900/2900
✅ Selected images downloaded. (Already-present files were skipped — safe to re-run.)
✅ Sample lists ready: train=2000, val=400, test=500


In [7]:
from google.colab import userdata

print("Attempting to retrieve HF_TOKEN from Colab Secrets...")
try:
    test_hf_token = userdata.get('HF_TOKEN')
    if test_hf_token:
        print("✅ HF_TOKEN successfully retrieved! (Value is hidden for security)")
        # You can uncomment the line below to see part of the token for debugging, but be careful not to share it
        # print(f"  First 5 chars of token: {test_hf_token[:5]}...")
    else:
        print("❌ HF_TOKEN is empty or not found, even though `userdata.get` didn't raise an error. Please ensure a value is set.")
except Exception as e:
    print(f"❌ Failed to retrieve HF_TOKEN. Error: {e}")
    print("Please re-check the 'Secrets' panel in the left sidebar to ensure 'HF_TOKEN' is named correctly and 'Notebook access' is ON.")


Attempting to retrieve HF_TOKEN from Colab Secrets...
✅ HF_TOKEN successfully retrieved! (Value is hidden for security)


### Debugging Hugging Face Token for `401 Unauthorized`

In [8]:
# Verify Hugging Face token status
from huggingface_hub import HfApi, get_full_repo_name, get_token
import os

token = get_token()
if token:
    if token == "hf_your_read_token":
        print('❌ Invalid Hugging Face token: The placeholder "hf_your_read_token" was used. Please replace it with your actual token in cell e03_05a_c.')
    else:
        print('✅ Hugging Face token found in environment/cache.')
        api = HfApi(token=token)
        try:
            user_info = api.whoami()
            print(f'Logged in as: {user_info["name"]}')

            # Check access to the specific repo
            repo_id = CRASHCAR_HF_REPO
            try:
                # Attempt to get repo info, which requires access for gated repos
                api.repo_info(repo_id=repo_id, repo_type='dataset', token=token)
                print(f'✅ Token has access to the dataset: {repo_id}')
            except Exception as e:
                print(f'❌ Token does NOT have access to the dataset {repo_id}: {e!r}')
                print("It's possible your token needs to be refreshed, or the dataset terms have not been accepted.")
                print('Please ensure you have accepted the dataset terms on the Hugging Face page (JensParslov/CrashCar) AND generate a new token (preferably a "write" token) and re-run notebook_login().')
        except Exception as e:
            print(f'❌ Failed to get user info with the found token: {e!r}')
            print("Your token might be invalid or expired. Please generate a new token and re-run notebook_login().")
else:
    print('❌ No Hugging Face token found. Please run `notebook_login()` again and ensure you provide a valid token.')


✅ Hugging Face token found in environment/cache.
Logged in as: lele22e
✅ Token has access to the dataset: JensParslov/CrashCar


If the above output indicates that the token does **not** have access, or no token was found, please follow these steps:

1.  Go to the [Hugging Face settings page](https://huggingface.co/settings/tokens).
2.  Generate a **new token** with **`write`** access (sometimes gated datasets require this to acknowledge terms, even for reading).
3.  In Colab, go to the "🔑" icon in the left sidebar, add a new secret, and name it `HF_TOKEN`. Paste your new token there.
4.  **Re-run cell `a99ce097` (where `notebook_login()` is located) and enter your new token.**
5.  **Re-run cell `e03_05a_c`** to attempt the download again.

In [9]:
# ── CrashCar101 (27 classes) → canonical ────────────────────────────────────
# SOURCED (see assumption #2, Section 0) from:
#  (a) cgpart_3d.zip -> CGPart/labels/car/*.json (5 car models; union = 31 named parts,
#      matching the paper's stated "31-category taxonomy" starting point), and
#  (b) CrashCar101's own generator, scene_class.py (self.bump_keys/window_keys/light_keys
#      hardcode these exact CGPart names; self.wheel_names = self.parts["wheel"] confirms
#      the 4 wheel parts are pre-merged into one "wheel" key, matching the paper's rule).
# Merges applied (per the paper, WACV24 §3): 4 wheel parts -> 1 'wheel'; 2 license-plate
# parts -> 1 'license_plate'. Mirrors are NOT merged (paper doesn't mention it, and CGPart's
# left_mirror/right_mirror are already singular per side). 11 (panels) + 8 (windows) +
# 4 (lights) + 1 (wheel) + 2 (mirrors) + 1 (plate) = 27, matching the paper's count exactly.
#
# Window classes (back_left_window etc.) are real, separately-rendered CrashCar101 classes —
# not the same mask as their door — but our canonical set has no separate glass-on-door part
# (labor_hours_lookup.py bills door glass as a 'glass' damage_type under the DOOR's own hours
# row), so they roll up to that door's canonical part, same as a real broken door window would
# be billed. Quarter windows and the merged license plate have no clean canonical target and
# are dropped. See assumption #2 for the fender gap: CrashCar101/CGPart has no fender class at
# all, so front/back_left/right_fender get zero training instances from this dataset.
CRASHCAR_TO_CANONICAL = {
    # panels / frame / body (bump_keys)
    'back_bumper':          'back_bumper',
    'front_bumper':         'front_bumper',
    'back_left_door':       'back_left_door',
    'back_right_door':      'back_right_door',
    'front_right_door':     'front_right_door',
    'front_left_door':      'front_left_door',
    'left_frame':           'left_sill',    # CGPart's door-sill/rocker-area class
    'right_frame':          'right_sill',
    'trunk':                'trunk',
    'roof':                 'roof',
    'hood':                 'hood',
    # glass (window_keys) — door windows roll up to their door; quarter windows dropped
    'back_left_window':     'back_left_door',
    'back_right_window':    'back_right_door',
    'front_left_window':    'front_left_door',
    'front_right_window':   'front_right_door',
    'left_quarter_window':  None,   # pillar-area glass, no clean canonical target -> drop
    'right_quarter_window': None,
    'front_windshield':     'front_glass',
    'back_windshield':      'back_glass',
    # lights (light_keys)
    'left_head_light':      'front_left_light',
    'right_head_light':     'front_right_light',
    'left_tail_light':      'back_left_light',
    'right_tail_light':     'back_right_light',
    # wheel (already merged 4->1 at the CrashCar101 source)
    'wheel':                'wheel',
    # mirrors (not merged at source)
    'left_mirror':           'left_mirror',
    'right_mirror':          'right_mirror',
    # plate (already merged 2->1 at the source per the paper) — no canonical target
    'license_plate':          None,
}
_cc_dropped = [k for k, v in CRASHCAR_TO_CANONICAL.items() if v is None]
_cc_bad = [v for v in CRASHCAR_TO_CANONICAL.values() if v is not None and v not in CANONICAL_INDEX]
assert not _cc_bad, f'CRASHCAR_TO_CANONICAL targets not in CANONICAL_CLASSES: {_cc_bad}'
assert len(CRASHCAR_TO_CANONICAL) == 27, f'expected 27 CrashCar101 source classes, got {len(CRASHCAR_TO_CANONICAL)}'
print(f'✅ CRASHCAR_TO_CANONICAL: {len(CRASHCAR_TO_CANONICAL)} source classes, '
      f'{len(CRASHCAR_TO_CANONICAL) - len(_cc_dropped)} kept, {len(_cc_dropped)} dropped: {_cc_dropped}')

_fenders_untouched = [c for c in CANONICAL_CLASSES if 'fender' in c]
print(f'⚠️  No CrashCar101 source maps to: {_fenders_untouched} — zero instances of these '
      f'canonical classes come from CrashCar101 in this experiment (see assumption #2).')


def validate_crashcar_mapping(discovered_names):
    """Call this after Section 5b's discovery step, once you know what names/ids the
    actually-downloaded dataset exposes. Confirms the 27-name taxonomy above still lines
    up with what's really there — catches a renamed/different release, doesn't invent one."""
    expected = set(CRASHCAR_TO_CANONICAL)
    found = set(discovered_names)
    missing = expected - found
    extra = found - expected
    if missing:
        print(f'⚠️  Expected CrashCar101 classes NOT found in the downloaded data: {sorted(missing)}')
    if extra:
        print(f'⚠️  Classes found in the downloaded data NOT in our expected 27: {sorted(extra)} '
              f'— add these to CRASHCAR_TO_CANONICAL (or map to None to drop) before Section 6.')
    if not missing and not extra:
        print('✅ Downloaded dataset\'s class names match the expected 27-class taxonomy exactly.')


✅ CRASHCAR_TO_CANONICAL: 27 source classes, 24 kept, 3 dropped: ['left_quarter_window', 'right_quarter_window', 'license_plate']
⚠️  No CrashCar101 source maps to: ['front_left_fender', 'front_right_fender', 'back_left_fender', 'back_right_fender'] — zero instances of these canonical classes come from CrashCar101 in this experiment (see assumption #2).


## 🌐 Section 5 — Acquire datasets

In [10]:
from huggingface_hub import notebook_login

notebook_login()

In [11]:
# ── 5b. Discover how CrashCar101 actually encodes its 'part' masks — READ THIS OUTPUT ──
# Two representations are plausible and this notebook handles both (see assumption #2):
#   'named'   — the sample's 'part' value is dict-like: {part_name: per-part mask array}.
#               Safe to use directly against CRASHCAR_TO_CANONICAL (name-keyed) regardless
#               of any id numbering.
#   'indexed' — the sample's 'part' value is a single-channel image where pixel value =
#               a numeric part id. This is what CrashCar101 actually ships (plain PNG masks
#               can't carry a 'named' dict). Section 5a already resolved and attached the
#               correct per-scene id->name legend (`SCENE_LEGENDS`, sourced from the
#               generator repo's data/accepted/<hash>.txt manifests — see cell 0) to every
#               sample's `part_value.info['legend']`, so this is safely usable, not guessed.
import numpy as np
from PIL import Image


def sample_rows(source, data, n=50, seed=SEED):
    rng = np.random.RandomState(seed)
    if source == 'hf_datasets':
        split = data['train'] if 'train' in data else list(data.values())[0]
        idxs = rng.choice(len(split), size=min(n, len(split)), replace=False)
        return [split[int(i)] for i in idxs]
    else:
        idxs = rng.choice(len(data), size=min(n, len(data)), replace=False)
        rows = []
        for i in idxs:
            rec = data[int(i)]
            part_img = Image.open(rec['part'])
            legend = SCENE_LEGENDS.get(rec.get('scene'))
            if legend is not None:
                part_img.info['legend'] = legend
            rows.append({'image': rec['image'], 'part': part_img})
        return rows


def inspect_part_repr(rows):
    part0 = rows[0]['part']
    if isinstance(part0, dict):
        names_seen = set()
        for r in rows:
            names_seen.update(r['part'].keys())
        return 'named', names_seen
    if hasattr(part0, 'mode') and part0.mode in ('L', 'P', 'I'):
        ids_seen = set()
        for r in rows:
            ids_seen.update(np.unique(np.array(r['part'])).tolist())
        return 'indexed', ids_seen
    return 'unknown', type(part0)


sample_rows_ = sample_rows(crashcar_source, crashcar_data)
part_repr_mode, part_repr_info = inspect_part_repr(sample_rows_)
print(f'✅ Detected \'part\' representation: {part_repr_mode}')

if part_repr_mode == 'named':
    print(f'   Part names seen across {len(sample_rows_)} sampled rows: {sorted(part_repr_info)}')
    validate_crashcar_mapping(part_repr_info)
elif part_repr_mode == 'indexed':
    print(f'   Discovered raw pixel-value ids (sampled rows only): {sorted(part_repr_info)}')
    # Validate against the FULL set of part names across every resolved scene legend (Section
    # 5a), not just the sampled rows — this is the real per-name check now that ids are
    # legend-backed, mirroring what the 'named' branch already does above.
    _all_legend_names = {name for legend in SCENE_LEGENDS.values() for name in legend.values()}
    print(f'   Part names across all {len(SCENE_LEGENDS)} resolved scene legends: {sorted(_all_legend_names)}')
    validate_crashcar_mapping(_all_legend_names)
    _n_with_legend = sum(1 for s in sample_rows_ if s['part'].info.get('legend') is not None)
    print(f'   {_n_with_legend}/{len(sample_rows_)} sampled rows carry an attached legend '
          '(should be all of them — Section 5a only kept scenes it could resolve).')
else:
    print(f'⚠️  Unrecognized \'part\' sample type: {part_repr_info} — inspect a raw sample manually '
          '(`sample_rows_[0][\'part\']`) before continuing; Section 6 only handles named-dict or indexed-image.')


✅ Detected 'part' representation: indexed
   Discovered raw pixel-value ids (sampled rows only): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
   Part names across all 92 resolved scene legends: ['back_bumper', 'back_left_door', 'back_left_window', 'back_license_plate', 'back_right_door', 'back_right_window', 'back_windshield', 'background', 'front_bumper', 'front_left_door', 'front_left_window', 'front_license_plate', 'front_right_door', 'front_right_window', 'front_windshield', 'hood', 'left_frame', 'left_head_light', 'left_mirror', 'left_quarter_window', 'left_tail_light', 'right_frame', 'right_head_light', 'right_mirror', 'right_quarter_window', 'right_tail_light', 'roof', 'trunk', 'wheel']
⚠️  Expected CrashCar101 classes NOT found in the downloaded data: ['license_plate']
⚠️  Classes found in the downloaded data NOT in our expected 27: ['back_license_plate', 'background', 'front_license_plate'] — add these to CRASHCAR_TO_CA

In [12]:
# ── 5c. Reuse exp02's frozen cropped-carparts dataset (do NOT rebuild it) ──
assert os.path.isdir(EXP02_DATASET_DIR), (
    f'{EXP02_DATASET_DIR} not found. Run exp02_finetune_cropped_carparts.ipynb (Sections 4-6) '
    'at least once first so its frozen v1_cropped_carparts dataset exists — this notebook '
    'reuses it rather than regenerating the crops, to avoid drift between the two experiments.'
)
import yaml
with open(os.path.join(EXP02_DATASET_DIR, 'carparts-crops.yaml')) as f:
    carparts_crops_yaml = yaml.safe_load(f)
CARPARTS_NAMES = carparts_crops_yaml['names']  # {id: name}, original 23 carparts-seg classes — unchanged by exp02
print(f'✅ Reusing exp02 dataset: {EXP02_DATASET_DIR}')
print(f'   {len(CARPARTS_NAMES)} classes: {list(CARPARTS_NAMES.values())}')
_missing = [n for n in CARPARTS_NAMES.values() if n not in CARPARTS_TO_CANONICAL]
assert not _missing, f'CARPARTS_TO_CANONICAL is missing classes actually present in exp02\'s dataset: {_missing}'


✅ Reusing exp02 dataset: /content/drive/MyDrive/carparts/segmentation/datasets/v1_cropped_carparts
   23 classes: ['back_bumper', 'back_door', 'back_glass', 'back_left_door', 'back_left_light', 'back_light', 'back_right_door', 'back_right_light', 'front_bumper', 'front_door', 'front_glass', 'front_left_door', 'front_left_light', 'front_light', 'front_right_door', 'front_right_light', 'hood', 'left_mirror', 'object', 'right_mirror', 'tailgate', 'trunk', 'wheel']


## ✂️ Section 6 — Convert CrashCar101 masks → canonical-class YOLO-seg polygons

Semantic masks aren't instance polygons. For each sampled image: for each canonical
class present in its `part` mask, take connected components of that class's pixels
as separate "instances," extract each component's contour, simplify it, and write a
normalized YOLO-seg polygon line. Small/noisy components are dropped (`MIN_INSTANCE_PX`).
Handles both representations discovered in Section 5b — see assumption #2 for why the
indexed-image path requires a per-sample legend rather than a fixed id order.


In [13]:
import cv2
from pathlib import Path


def _instances_from_binary_mask(binary, canon_idx, min_instance_px, epsilon_frac=0.01):
    h, w = binary.shape
    lines = []
    n_components, labels = cv2.connectedComponents(binary.astype('uint8'))
    for comp_id in range(1, n_components):
        comp_mask = (labels == comp_id).astype('uint8')
        if comp_mask.sum() < min_instance_px:
            continue
        contours, _ = cv2.findContours(comp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        contour = max(contours, key=cv2.contourArea)
        perimeter = cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, epsilon_frac * perimeter, True)
        if len(approx) < 3:
            continue
        coords = []
        for pt in approx.reshape(-1, 2):
            coords.extend([min(1.0, max(0.0, pt[0] / w)), min(1.0, max(0.0, pt[1] / h))])
        lines.append(f'{canon_idx} ' + ' '.join(f'{c:.6f}' for c in coords))
    return lines


def named_masks_to_yolo_polygons(part_dict, name_to_canonical, min_instance_px=64):
    """part_dict: {part_name: mask-like array/PIL Image}. Safe regardless of any id
    ordering — matched purely by name against CRASHCAR_TO_CANONICAL."""
    lines = []
    for name, canon in name_to_canonical.items():
        if canon is None or name not in part_dict:
            continue
        arr = np.asarray(part_dict[name])
        binary = (arr > 0) if arr.dtype != bool else arr
        if binary.sum() < min_instance_px:
            continue
        lines.extend(_instances_from_binary_mask(binary, CANONICAL_INDEX[canon], min_instance_px))
    return lines


def indexed_mask_to_yolo_polygons(mask_ids, id_to_name_legend, name_to_canonical, min_instance_px=64):
    """mask_ids: 2D array of pixel-value ids. id_to_name_legend: THIS SAMPLE's id->name
    (per assumption #2, ids are per-3D-model, not global — never pass a guessed global one)."""
    lines = []
    for src_id, name in id_to_name_legend.items():
        canon = name_to_canonical.get(name)
        if canon is None:
            continue
        binary = (mask_ids == src_id)
        if binary.sum() < min_instance_px:
            continue
        lines.extend(_instances_from_binary_mask(binary, CANONICAL_INDEX[canon], min_instance_px))
    return lines


MIN_INSTANCE_PX = 64
print('✅ Mask-to-polygon conversion defined (named-dict and legend-backed-indexed paths).')


✅ Mask-to-polygon conversion defined (named-dict and legend-backed-indexed paths).


In [14]:
# ── Refs into the already-split, already-subsampled sample lists from Section 5a ──
# Splitting + subsampling now happens BEFORE download (Section 5a) so we never fetch more
# image bytes than Section 3's CRASHCAR_SAMPLE_FRAC/CRASHCAR_MAX_IMAGES actually need. This
# cell just wires up the (data, refs) pairs Section 7's convert_crashcar_split expects.
assert part_repr_mode in ('named', 'indexed'), (
    f"'part' representation is '{part_repr_mode}' (Section 5b) — only 'named' or a legend-backed "
    "'indexed' representation are handled. Resolve this before continuing (see Section 5b output)."
)
if part_repr_mode == 'indexed':
    print('⚠️  Indexed representation detected — each sample MUST carry its own id->name legend '
          '(assumption #2). If your loaded rows do not expose one, CrashCar101 cannot be safely '
          'converted; stop here and proceed with cropped-carparts alone rather than guess an id order.')

cc_train_refs = list(range(len(crashcar_train_data)))
cc_val_refs = list(range(len(crashcar_val_data)))
cc_test_refs = list(range(len(crashcar_test_data)))
print(f'✅ Refs ready: train={len(cc_train_refs)}, val={len(cc_val_refs)}, test={len(cc_test_refs)}')


⚠️  Indexed representation detected — each sample MUST carry its own id->name legend (assumption #2). If your loaded rows do not expose one, CrashCar101 cannot be safely converted; stop here and proceed with cropped-carparts alone rather than guess an id order.
✅ Refs ready: train=2000, val=400, test=500


In [ ]:
def load_sample(source, data, ref):
    if source == 'hf_datasets':
        split_name, row_idx = ref
        row = data[split_name][row_idx]
        return row['image'], row['part']
    else:
        rec = data[ref]
        img = Image.open(rec['image'])
        part_img = Image.open(rec['part'])
        legend = SCENE_LEGENDS.get(rec.get('scene'))  # resolved in Section 5a, keyed by scene hash
        if legend is not None:
            part_img.info['legend'] = legend
        return img, part_img


def ref_stem(ref):
    return f'crashcar_{ref[0]}_{ref[1]:07d}' if isinstance(ref, tuple) else f'crashcar_{ref:07d}'


def convert_one_sample(img, part_value):
    """Dispatches on the representation discovered in Section 5b. Returns YOLO-seg
    label lines, or None if this sample can't be converted (indexed w/o legend)."""
    if part_repr_mode == 'named':
        return named_masks_to_yolo_polygons(part_value, CRASHCAR_TO_CANONICAL, MIN_INSTANCE_PX)
    # indexed: only proceed if this specific row carries its own legend
    legend = part_value.info.get('legend') if hasattr(part_value, 'info') else None
    if legend is None:
        return None  # no per-sample legend -> unsafe to interpret ids, skip (see assumption #2)
    mask_ids = np.array(part_value.convert('L') if part_value.mode not in ('L', 'P', 'I') else part_value)
    return indexed_mask_to_yolo_polygons(mask_ids, legend, CRASHCAR_TO_CANONICAL, MIN_INSTANCE_PX)


def convert_crashcar_split(source, data, refs, out_images_dir, out_labels_dir):
    os.makedirs(out_images_dir, exist_ok=True)
    os.makedirs(out_labels_dir, exist_ok=True)
    n_written, n_empty, n_no_legend = 0, 0, 0
    for ref in refs:
        img, part_value = load_sample(source, data, ref)
        lines = convert_one_sample(img, part_value)
        if lines is None:
            n_no_legend += 1
            continue
        if not lines:
            n_empty += 1
            continue
        stem = ref_stem(ref)
        img.convert('RGB').save(os.path.join(out_images_dir, f'{stem}.jpg'), quality=95)
        with open(os.path.join(out_labels_dir, f'{stem}.txt'), 'w') as f:
            f.write('\n'.join(lines) + '\n')
        n_written += 1
    return {'written': n_written, 'skipped_no_canonical_instances': n_empty, 'skipped_no_legend': n_no_legend}


cc_train_out_img = os.path.join(LOCAL_BUILD_ROOT, 'train', 'images')
cc_train_out_lbl = os.path.join(LOCAL_BUILD_ROOT, 'train', 'labels')
cc_val_out_img = os.path.join(LOCAL_BUILD_ROOT, 'val', 'images')
cc_val_out_lbl = os.path.join(LOCAL_BUILD_ROOT, 'val', 'labels')
cc_test_out_img = os.path.join(LOCAL_BUILD_ROOT, 'test', 'images')
cc_test_out_lbl = os.path.join(LOCAL_BUILD_ROOT, 'test', 'labels')

print('▶ Converting CrashCar101 train subsample...')
crashcar_train_stats = convert_crashcar_split(crashcar_source, crashcar_train_data, cc_train_refs, cc_train_out_img, cc_train_out_lbl)
print(f'   {crashcar_train_stats}')
print('▶ Converting CrashCar101 val subsample...')
crashcar_val_stats = convert_crashcar_split(crashcar_source, crashcar_val_data, cc_val_refs, cc_val_out_img, cc_val_out_lbl)
print(f'   {crashcar_val_stats}')
print('▶ Converting CrashCar101 test subsample (held out — never trained/validated on)...')
crashcar_test_stats = convert_crashcar_split(crashcar_source, crashcar_test_data, cc_test_refs, cc_test_out_img, cc_test_out_lbl)
print(f'   {crashcar_test_stats}')


▶ Converting CrashCar101 train subsample...


## 🔁 Section 7 — Remap exp02's cropped-carparts labels into canonical class space

In [ ]:
def remap_carparts_split(src_images_dir, src_labels_dir, out_images_dir, out_labels_dir):
    """Same polygon geometry (already YOLO-seg from exp02) — only the class id column
    changes. Instances whose class has no canonical target (CARPARTS_TO_CANONICAL[name]
    is None) are dropped, per your instructions."""
    os.makedirs(out_images_dir, exist_ok=True)
    os.makedirs(out_labels_dir, exist_ok=True)
    n_written, n_dropped_instances, n_skipped_empty = 0, 0, 0
    for img_path in sorted(Path(src_images_dir).glob('*.jpg')):
        label_path = Path(src_labels_dir) / f'{img_path.stem}.txt'
        if not label_path.exists():
            continue
        kept_lines = []
        for line in label_path.read_text().strip().splitlines():
            if not line.strip():
                continue
            parts = line.split()
            src_name = CARPARTS_NAMES[int(parts[0])]
            canon = CARPARTS_TO_CANONICAL.get(src_name)
            if canon is None:
                n_dropped_instances += 1
                continue
            kept_lines.append(f'{CANONICAL_INDEX[canon]} ' + ' '.join(parts[1:]))
        if not kept_lines:
            n_skipped_empty += 1
            continue
        shutil_copy_or_link(img_path, os.path.join(out_images_dir, img_path.name))
        with open(os.path.join(out_labels_dir, f'{img_path.stem}.txt'), 'w') as f:
            f.write('\n'.join(kept_lines) + '\n')
        n_written += 1
    return {'written': n_written, 'dropped_instances': n_dropped_instances, 'skipped_empty_images': n_skipped_empty}


import shutil

def shutil_copy_or_link(src, dst):
    if os.path.exists(dst):
        return
    try:
        os.symlink(os.path.abspath(src), dst)  # avoid duplicating exp02's crop bytes on local disk
    except OSError:
        shutil.copy2(src, dst)


print('▶ Remapping exp02 cropped-carparts train split...')
carparts_train_stats = remap_carparts_split(
    os.path.join(EXP02_DATASET_DIR, 'train', 'images'), os.path.join(EXP02_DATASET_DIR, 'train', 'labels'),
    cc_train_out_img, cc_train_out_lbl,
)
print(f'   {carparts_train_stats}')
print('▶ Remapping exp02 cropped-carparts val split...')
carparts_val_stats = remap_carparts_split(
    os.path.join(EXP02_DATASET_DIR, 'val', 'images'), os.path.join(EXP02_DATASET_DIR, 'val', 'labels'),
    cc_val_out_img, cc_val_out_lbl,
)
print(f'   {carparts_val_stats}')


## 🧬 Section 8 — Leak-safe assembly, dedup check, and `data.yaml`

In [ ]:
# ── Exact-duplicate check (not full pairwise near-dup search — see rationale) ──
# CrashCar101 is procedurally synthetic and cropped-carparts is derived from
# real-photo carparts-seg crops -> cross-dataset duplication is not a realistic risk.
# Within-dataset leakage is already prevented upstream: carparts-seg's own train/val
# split is disjoint (exp02 built crops split-by-split from it), and CrashCar101's
# split (Section 6b) is drawn from disjoint index pools too. A full O(n^2) perceptual
# hash over ~12k+ images would be slow and wouldn't meaningfully reduce risk beyond
# this. What IS checked cheaply: exact byte-identical files ending up on both sides
# of train/val (e.g. a bug re-including a file), via MD5 of file bytes.
import hashlib


def file_md5(path, chunk=65536):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()


def hashes_for(images_dir):
    return {file_md5(p): p.name for p in Path(images_dir).glob('*.jpg')}


split_hashes = {
    'train': hashes_for(cc_train_out_img),
    'val': hashes_for(cc_val_out_img),
    'test': hashes_for(cc_test_out_img),
}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    overlap = set(split_hashes[a]) & set(split_hashes[b])
    if overlap:
        raise RuntimeError(
            f'{len(overlap)} byte-identical images appear in BOTH {a} and {b}: '
            f'{[split_hashes[a][h] for h in list(overlap)[:5]]} ... — fix before training/evaluating. '
            f'{b} in particular must be leak-free (Section 11 reports metrics on it).'
        )
print('✅ No exact-duplicate images across train/val/test ('
      f"{len(split_hashes['train'])} train, {len(split_hashes['val'])} val, {len(split_hashes['test'])} test hashed).")


In [ ]:
import yaml # Import yaml module explicitly to prevent error after changes

# __ data.yaml — canonical class names/order, matching labor_hours_lookup.py exactly __
local_yaml_path = os.path.join(LOCAL_BUILD_ROOT, 'unified-canonical.yaml')
yaml_data = {
    'path': LOCAL_BUILD_ROOT,
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',  # CrashCar101-only, held out — used by Section 11's model.val(split='test')
    'names': {i: name for i, name in enumerate(CANONICAL_CLASSES)},
}
with open(local_yaml_path, 'w') as f:
    yaml.dump(yaml_data, f, sort_keys=False)
print(f'✅ Saved training data yaml → {local_yaml_path}')
print(f'   {len(CANONICAL_CLASSES)} canonical classes: {CANONICAL_CLASSES}')

# Define split_kind based on the context that a fixed 70/15/15 split is used.
split_kind = 'fixed_seed_70_15_15'

# Archive the frozen, versioned unified dataset to Drive (docs §7) — same pattern as exp02.
if os.path.exists(DRIVE_DATASETS_DIR):
    print(f'⚠️ {DRIVE_DATASETS_DIR} already exists — bump DATASET_VERSION in Section 3 instead of overwriting.')
else:
    print('📦 Archiving unified dataset to Drive (labels only + a manifest; images are large — see note)...')
    # NOTE: this copies labels + yaml + a manifest, not the raw images themselves (images/
    # can be regenerated deterministically from CRASHCAR_* + SEED + exp02's own already-archived
    # dataset; archiving them again would roughly double CrashCar101's footprint on Drive).
    # If you want the images archived too, uncomment the two shutil.copytree lines below.
    os.makedirs(DRIVE_DATASETS_DIR, exist_ok=True)
    for split in ['train', 'val', 'test']:
        shutil.copytree(os.path.join(LOCAL_BUILD_ROOT, split, 'labels'),
                         os.path.join(DRIVE_DATASETS_DIR, split, 'labels'))
        # shutil.copytree(os.path.join(LOCAL_BUILD_ROOT, split, 'images'),
        #                  os.path.join(DRIVE_DATASETS_DIR, split, 'images'))
    drive_yaml = dict(yaml_data)
    with open(os.path.join(DRIVE_DATASETS_DIR, 'unified-canonical.yaml'), 'w') as f:
        yaml.dump(drive_yaml, f, sort_keys=False)
    dataset_info = {
        'sources': {
            'crashcar101': {'repo': CRASHCAR_HF_REPO, 'split_kind': split_kind,
                             'train_images': crashcar_train_stats, 'val_images': crashcar_val_stats,
                             'test_images': crashcar_test_stats,
                             'sample_frac': CRASHCAR_SAMPLE_FRAC, 'max_images': CRASHCAR_MAX_IMAGES,
                             'test_sample_frac': CRASHCAR_TEST_SAMPLE_FRAC, 'test_max_images': CRASHCAR_TEST_MAX_IMAGES},
            'cropped_carparts_exp02': {'dir': EXP02_DATASET_DIR,
                                        'train': carparts_train_stats, 'val': carparts_val_stats},
        },
        'canonical_classes': CANONICAL_CLASSES,
        'seed': SEED,
        'built_for_experiment': EXP_ID,
    }
    with open(os.path.join(DRIVE_DATASETS_DIR, 'dataset_info.yaml'), 'w') as f:
        yaml.dump(dataset_info, f, sort_keys=False)
    print(f'✅ Archived versioned dataset → {DRIVE_DATASETS_DIR}')

In [ ]:
import os

print(f'Contents of {DRIVE_DATASETS_DIR}:')
if os.path.exists(DRIVE_DATASETS_DIR):
    for root, dirs, files in os.walk(DRIVE_DATASETS_DIR):
        level = root.replace(DRIVE_DATASETS_DIR, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f'{subindent}{f}')
else:
    print(f'The directory {DRIVE_DATASETS_DIR} does not exist. The archiving process may not have run or completed successfully.')

## 🏋️ Section 9 — Train: frozen-backbone warmup, then unfrozen with early stopping

Two sequential `model.train()` calls (assumption #7): Phase 1 freezes the first
`FREEZE_LAYERS` layers for `FREEZE_EPOCHS` epochs; Phase 2 resumes from Phase 1's
`last.pt` fully unfrozen, with `patience`-based early stopping and the augmentation
settings from Section 3.


In [ ]:
import os # Needed for os.path.join
from ultralytics import YOLO

# Redefine variables from previous cells to ensure scope
MODEL_SOURCE   = '/content/drive/MyDrive/carparts/weights/best.pt'
IMG_SIZE       = 320
FREEZE_EPOCHS  = 10
FREEZE_LAYERS  = 10
SEED           = 42
WEIGHT_DECAY   = 0.001
EXP_ID         = 'exp03_crashcar_plus_carparts'
RUNS_PROJECT   = '/content/runs_exp03'
LOCAL_BUILD_ROOT = '/content/unified_build'
local_yaml_path = os.path.join(LOCAL_BUILD_ROOT, 'unified-canonical.yaml')

# Augmentation parameters
AUGMENT = dict(
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,   # color jitter — library defaults
    degrees=0.0, translate=0.1, scale=0.5, shear=0.0,  # geometric — library defaults
    fliplr=0.5, flipud=0.0,              # horizontal flip only (vertical flip is not physically meaningful for a car)
    mosaic=1.0,                          # on — library default
    mixup=0.0, copy_paste=0.0,           # off — see note above
)

# Check if the pretrained model exists before loading
if not os.path.exists(MODEL_SOURCE):
    raise FileNotFoundError(
        f"Pretrained model '{MODEL_SOURCE}' not found. "
        "Please ensure the 'best.pt' file from exp01/exp02 is uploaded to "
        "'/content/drive/MyDrive/carparts/weights/' in your Google Drive, "
        "or update the `MODEL_SOURCE` variable to the correct path if it's located elsewhere."
    )

model = YOLO(MODEL_SOURCE)  # SAME pretrained checkpoint exp01/exp02 evaluated
phase1 = model.train(
    data=local_yaml_path,
    imgsz=IMG_SIZE,
    epochs=FREEZE_EPOCHS,
    freeze=FREEZE_LAYERS,
    seed=SEED,
    weight_decay=WEIGHT_DECAY,
    project=RUNS_PROJECT,
    name=f'{EXP_ID}_phase1_frozen',
    exist_ok=True,
    batch=8, # Added for resource management
    **AUGMENT,
)
print('✅ Phase 1 (frozen backbone warmup) complete.')

### Debugging `MODEL_SOURCE` `FileNotFoundError`

It seems the `best.pt` file is still not being found at the path specified by `MODEL_SOURCE`. This can happen if the file is shared with you but not directly in your 'My Drive', or if there's a slight mismatch in the directory path (e.g., case sensitivity).

Let's check the contents of the expected directory to diagnose this. Please execute the following cell.

In [ ]:
# Verify the Google Drive mount and the exact path to MODEL_SOURCE
import os

# Check if Google Drive is mounted
if not os.path.exists('/content/drive/MyDrive'):
    print("❌ Google Drive does not appear to be mounted. Please run cell `e03_01_c` to mount your Drive.")
else:
    print("✅ Google Drive is mounted.")

# Define the expected directory for the weights
expected_weights_dir = os.path.dirname(MODEL_SOURCE)

print(f"\nAttempting to list contents of: {expected_weights_dir}")

if os.path.exists(expected_weights_dir):
    print(f"✅ Directory '{expected_weights_dir}' exists. Contents:")
    for item in os.listdir(expected_weights_dir):
        print(f"  - {item}")
    if os.path.exists(MODEL_SOURCE):
        print(f"\n✅ The file '{os.path.basename(MODEL_SOURCE)}' is found in '{expected_weights_dir}'.")
        print("This suggests the error might be elsewhere, or a temporary Colab glitch.")
        print("Please try re-running cell `e03_09a_c` again.")
    else:
        print(f"\n❌ The file '{os.path.basename(MODEL_SOURCE)}' is NOT found in '{expected_weights_dir}'.")
        print("Please ensure the file is located at this exact path. If it's a shared file, you may need to add a shortcut to it in your 'My Drive' from the Google Drive interface.")
        print("Instructions for adding a shortcut from 'Shared with me' to 'My Drive':")
        print("1. Go to drive.google.com.")
        print("2. In the left panel, click 'Shared with me'.")
        print("3. Locate the 'carparts' folder (or the 'weights' folder, or even 'best.pt' directly if possible).")
        print("4. Right-click on the folder/file.")
        print("5. Select 'Add shortcut to Drive'.")
        print("6. Choose 'My Drive' as the location and click 'Add shortcut'.")
        print("After creating the shortcut, the file should appear under `/content/drive/MyDrive/carparts/weights/best.pt` in Colab.")
        print("You may also need to restart the runtime (Runtime -> Restart runtime) after adding the shortcut for Colab to recognize it, then re-run cells from `e03_01_c` onwards.")
else:
    print(f"❌ Directory '{expected_weights_dir}' does NOT exist.")
    print("Please check the path in `MODEL_SOURCE` and ensure the `carparts/weights/` directory exists directly under your 'My Drive' in Google Drive.")

In [ ]:
!pip install -q ultralytics # Ensure ultralytics is installed
import os # Import os module
from ultralytics import YOLO # Import YOLO for model definition

# Redefine variables from previous cells to ensure scope
DRIVE_ROOT = '/content/drive/MyDrive/carparts/segmentation'
EXP_ID     = 'exp03_crashcar_plus_carparts'
EXP_DIR    = os.path.join(DRIVE_ROOT, 'experiments', EXP_ID)
RUNS_PROJECT = '/content/runs_exp03'

phase1_last = os.path.join(RUNS_PROJECT, f'{EXP_ID}_phase1_frozen', 'weights', 'last.pt')
model = YOLO(phase1_last)  # resume from the warmup phase, now unfrozen
phase2 = model.train(
    data=local_yaml_path,
    imgsz=IMG_SIZE,
    epochs=FINETUNE_EPOCHS,
    patience=PATIENCE,       # early stopping — stop when val mAP stalls
    seed=SEED,
    weight_decay=WEIGHT_DECAY,
    project=RUNS_PROJECT,
    name=f'{EXP_ID}_phase2_finetune',
    exist_ok=True,
    batch=8, # Added for resource management
    **AUGMENT,
)
print('✅ Phase 2 (unfrozen, early-stopped) complete.')

In [ ]:
run_dir = os.path.join(RUNS_PROJECT, f'{EXP_ID}_phase2_finetune')
weights_dir = os.path.join(EXP_DIR, 'weights')
os.makedirs(weights_dir, exist_ok=True)

for fname in ['best.pt', 'last.pt']:
    src = os.path.join(run_dir, 'weights', fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(weights_dir, fname))

for fname in ['results.csv', 'results.png']:
    src = os.path.join(run_dir, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(EXP_DIR, fname))

print(f'✅ Training complete. Weights + curves copied to {EXP_DIR}')


## 🖼️ Section 10 — Evaluate on the REAL close-ups: detection rate (exp02-comparable)

Reuses exp01/exp02's exact detection-rate logic so the number is directly comparable.


In [ ]:
FT_MODEL_PATH = os.path.join(EXP_DIR, 'weights', 'best.pt')
model = YOLO(FT_MODEL_PATH)
model.to(DEVICE)
CLASS_NAMES = model.names
print(f'✅ Fine-tuned model loaded for evaluation: {FT_MODEL_PATH}')
print(f'   Classes ({len(CLASS_NAMES)}): {list(CLASS_NAMES.values())}')

from collections import Counter


def list_images(folder):
    p = Path(folder)
    if not p.is_dir():
        raise FileNotFoundError(f'Expected subfolder not found: {folder}')
    return sorted([f for f in p.iterdir() if f.suffix.lower() in IMG_EXTS])


def run_inference(subfolder, paths):
    records = []
    preds_dir = os.path.join(EXP_DIR, 'val_preds', subfolder)
    for img_path in paths:
        result = model.predict(
            source=str(img_path), conf=CONF_THRESH, iou=IOU_THRESH,
            imgsz=IMG_SIZE, device=DEVICE, verbose=False
        )[0]
        boxes = result.boxes
        num_dets = 0 if boxes is None else len(boxes)
        classes = [] if num_dets == 0 else [CLASS_NAMES[int(c)] for c in boxes.cls]
        confs   = [] if num_dets == 0 else [float(c) for c in boxes.conf]
        top1 = classes[int(np.argmax(confs))] if confs else None

        annotated = result.plot()
        out_path = os.path.join(preds_dir, f'{img_path.stem}_pred.jpg')
        cv2.imwrite(out_path, annotated)

        records.append({
            'subfolder': subfolder, 'image': img_path.name, 'num_detections': num_dets,
            'detected_classes': ';'.join(classes), 'top1_class': top1,
            'max_conf': max(confs) if confs else 0.0, 'mean_conf': (sum(confs) / len(confs)) if confs else 0.0,
            'overlay_path': out_path,
        })
    return records


def summarize(subfolder_df, subfolder_name):
    total = len(subfolder_df)
    detected = int((subfolder_df['num_detections'] > 0).sum())
    detection_rate = 100.0 * detected / total if total else 0.0
    instance_counts = Counter()
    for classes_str in subfolder_df['detected_classes']:
        instance_counts.update([c for c in classes_str.split(';') if c])
    return {
        'subfolder': subfolder_name, 'total_images': total, 'detected_images': detected,
        'zero_detection_images': total - detected, 'detection_rate_pct': round(detection_rate, 2),
        'instance_class_distribution': dict(instance_counts),
    }


image_paths = {sub: list_images(os.path.join(REAL_DATA_ROOT, sub)) for sub in SUBFOLDERS}
all_records = []
for sub, paths in image_paths.items():
    print(f'▶ Running inference on "{sub}/" ({len(paths)} images)...')
    all_records.extend(run_inference(sub, paths))

per_image_df = pd.DataFrame(all_records)
summary_rows = [summarize(per_image_df[per_image_df['subfolder'] == sub], sub) for sub in SUBFOLDERS]
for row in summary_rows:
    print(f"\n📊 {row['subfolder']}/  detection rate: {row['detection_rate_pct']}% "
          f"({row['detected_images']}/{row['total_images']})")
    print(f"   Class distribution: {row['instance_class_distribution']}")

results_csv = os.path.join(EXP_DIR, 'real_eval_results.csv')
per_image_df.to_csv(results_csv, index=False)
summary_csv = os.path.join(EXP_DIR, 'real_eval_summary.csv')
pd.DataFrame([{k: v for k, v in r.items() if not isinstance(v, dict)} for r in summary_rows]).to_csv(summary_csv, index=False)
print(f'\n✅ Saved → {results_csv}\n✅ Saved → {summary_csv}')


## 🎯 Section 11 — Quantitative part-classification metrics (held-out CrashCar101 TEST split)

**Read assumption #5 before trusting this number.** This measures part-classification
skill *within the synthetic domain* — the `test/` split (Section 6b/8) is real CrashCar101
ground truth, never touched by train/val, so it's a legitimate, leak-free, no-manual-labeling
metric. It does **not** measure real-CrashLens transfer — that stays qualitative (Section 12).
Uses Ultralytics' own `model.val(split='test')`, which also auto-saves a confusion matrix.


In [ ]:
TEST_MODEL_PATH = os.path.join(EXP_DIR, 'weights', 'best.pt')
test_eval_model = YOLO(TEST_MODEL_PATH)

test_metrics = test_eval_model.val(
    data=local_yaml_path, split='test', imgsz=IMG_SIZE, conf=CONF_THRESH, iou=IOU_THRESH,
    device=DEVICE, project=EXP_DIR, name='crashcar_test_eval', plots=True, exist_ok=True,
)
print('✅ Evaluated on the held-out CrashCar101 test split.')
print(f'   Full Ultralytics output (incl. auto-saved confusion_matrix.png/.csv): '
      f"{os.path.join(EXP_DIR, 'crashcar_test_eval')}")


In [ ]:
# Per-class precision/recall/mAP — defensive extraction, since exact attribute names have
# shifted across Ultralytics versions; falls back to the always-stable results_dict if the
# detailed per-class arrays aren't where expected.
per_class_rows = []
try:
    seg = test_metrics.seg
    for i, cls_id in enumerate(seg.ap_class_index):
        cls_id = int(cls_id)
        per_class_rows.append({
            'canonical_class': CANONICAL_CLASSES[cls_id],
            'precision': float(seg.p[i]),
            'recall': float(seg.r[i]),
            'mAP50': float(seg.ap50[i]),
            'mAP50-95': float(seg.maps[cls_id]),
        })
    per_class_df = pd.DataFrame(per_class_rows).sort_values('canonical_class').reset_index(drop=True)
    tested_classes = set(per_class_df['canonical_class'])
    untested_classes = [c for c in CANONICAL_CLASSES if c not in tested_classes]
    print(f'✅ Per-class metrics for {len(per_class_df)}/{len(CANONICAL_CLASSES)} canonical classes '
          f'(the rest have zero instances in the CrashCar101 test split — expected for the fenders, '
          f'see assumption #2): {untested_classes}')
    print(per_class_df.to_string(index=False))
except Exception as e:
    print(f'⚠️  Per-class extraction failed ({e!r}) — Ultralytics API may have changed. '
          f'Falling back to aggregate results_dict:')
    print(test_metrics.results_dict)
    per_class_df = pd.DataFrame()

per_class_csv = os.path.join(EXP_DIR, 'crashcar_test_per_class_metrics.csv')
per_class_df.to_csv(per_class_csv, index=False)
print(f'✅ Saved → {per_class_csv}')


## 📈 Section 12 — Real CrashLens: detection rate before/after, and manual qualitative review

Detection rate (quantitative, exp02-comparable, no GT needed) plus **guided manual review**
for part classification: per assumption #5, this is a judgment call, not an automated score —
but it doesn't have to be a blind one. This surfaces the specific exp02 `cropped/` images that
predicted `hood` (the pattern you flagged) so you can check exp03's overlay for those exact
images first, then spot-check the rest. Record your verdict in `notes.md` (Section 13).


In [ ]:
EXP02_DIR = os.path.join(DRIVE_ROOT, 'experiments', 'exp02_finetune_cropped_carparts')
EXP02_REFERENCE_DETECTION = {
    'cropped': 55.0,  # exp02 detection rate on real close-ups — per your message / exp02's real_eval_summary.csv
    'full': 100.0,    # exp02 detection rate on real full-car shots
}  # NOTE: update from EXP02_DIR/real_eval_summary.csv if it differs

detection_rows = []
for r in summary_rows:
    sub = r['subfolder']
    before, after = EXP02_REFERENCE_DETECTION.get(sub), r['detection_rate_pct']
    delta = None if before is None else round(after - before, 2)
    detection_rows.append({'subfolder': sub, 'exp02_detection_rate_pct': before, 'exp03_detection_rate_pct': after, 'delta_pct': delta})
    print(f"   {sub:8s} | exp02: {before}%  ->  exp03: {after}%  (delta {delta:+.2f} pts)" if before is not None
          else f"   {sub:8s} | exp03: {after}%  (no exp02 reference)")
detection_comparison_df = pd.DataFrame(detection_rows)
detection_comparison_df.to_csv(os.path.join(EXP_DIR, 'exp02_vs_exp03_detection.csv'), index=False)


In [ ]:
# ── Manual review candidates, generated from exp02's own saved predictions (no GT needed) ──
print('Manual review guidance — no ground-truth labeling required:')
print('exp02 predicted "hood" on some cropped/ images where the true damage was reportedly')
print('glass — a concrete, checkable pattern to look for first, before a broader spot-check.\n')

exp02_results_csv = os.path.join(EXP02_DIR, 'real_eval_results.csv')
review_candidates = []
if os.path.exists(exp02_results_csv):
    exp02_pred_df = pd.read_csv(exp02_results_csv)
    candidates = exp02_pred_df[(exp02_pred_df['subfolder'] == 'cropped') &
                                (exp02_pred_df['detected_classes'].str.contains('hood', na=False))]
    print(f'{len(candidates)} exp02 cropped/ images had "hood" among their predictions:')
    for _, row in candidates.iterrows():
        exp03_overlay = os.path.join(EXP_DIR, 'val_preds', 'cropped', f"{Path(row['image']).stem}_pred.jpg")
        exists = os.path.exists(exp03_overlay)
        review_candidates.append({'image': row['image'], 'exp02_predicted': row['detected_classes'],
                                   'exp02_overlay': row['overlay_path'], 'exp03_overlay': exp03_overlay})
        print(f"  {row['image']}: exp02 predicted [{row['detected_classes']}]")
        print(f"    exp02 overlay: {row['overlay_path']}")
        print(f"    exp03 overlay: {exp03_overlay}  {'' if exists else '(MISSING -- check filename/path)'}")
else:
    print(f'{exp02_results_csv} not found -- open val_preds/cropped/ for both experiments and compare directly.')

review_candidates_csv = os.path.join(EXP_DIR, 'manual_review_candidates.csv')
pd.DataFrame(review_candidates).to_csv(review_candidates_csv, index=False)
print(f'\n✅ Saved candidate list → {review_candidates_csv}')
print('\nAlso spot-check a handful of full/ and cropped/ overlays beyond this candidate list, for')
print('wrong-part errors this pattern-match would not catch. Record verdicts in notes.md (Section 13)')
print('under Strengths/Weaknesses -- e.g. "glass on the N candidate images above now predicts')
print('front_glass/back_glass correctly in exp03" or "still predicts hood, unchanged".')


## 🗒️ Section 13 — Save `config.yaml` and `notes.md` (docs §8 template)

In [ ]:
from datetime import date

train_metrics_final = {}
results_csv_path = os.path.join(EXP_DIR, 'results.csv')
if os.path.exists(results_csv_path):
    _tdf = pd.read_csv(results_csv_path)
    _tdf.columns = [c.strip() for c in _tdf.columns]
    last = _tdf.iloc[-1].to_dict()
    train_metrics_final = {k: v for k, v in last.items() if any(t in k for t in ('mAP', 'precision', 'recall'))}

config = {
    'experiment_id': EXP_ID,
    'stage': 'Stage 2 — fine-tune (TWO variables vs exp02: +CrashCar101 data, +overfitting-prevention recipe)',
    'reference_experiment': 'exp02_finetune_cropped_carparts',
    'model_source': MODEL_SOURCE,
    'seed': SEED,
    'canonical_classes': CANONICAL_CLASSES,
    'class_mapping': {'carparts_to_canonical': CARPARTS_TO_CANONICAL, 'crashcar_to_canonical': CRASHCAR_TO_CANONICAL},
    'datasets': {
        'crashcar101': {'repo': CRASHCAR_HF_REPO, 'sample_frac': CRASHCAR_SAMPLE_FRAC, 'max_images': CRASHCAR_MAX_IMAGES,
                         'split_kind': split_kind, 'train_stats': crashcar_train_stats, 'val_stats': crashcar_val_stats},
        'cropped_carparts_exp02': {'dir': EXP02_DATASET_DIR, 'train_stats': carparts_train_stats, 'val_stats': carparts_val_stats},
    },
    'dataset_version': DATASET_VERSION,
    'splits': {'train': 'CrashCar101(train subsample) + cropped-carparts(train)',
               'val': 'CrashCar101(held-out subsample) + cropped-carparts(val) -- SYNTHETIC-LEANING, see notes.md limitation',
               'test': 'CrashCar101 held-out TEST subsample (official split where available) -- never trained/validated '
                       'on; used ONLY for Section 11 quantitative metrics. NOT real CrashLens images.'},
    'training': {
        'freeze_layers': FREEZE_LAYERS, 'freeze_epochs': FREEZE_EPOCHS, 'finetune_epochs': FINETUNE_EPOCHS,
        'patience': PATIENCE, 'weight_decay': WEIGHT_DECAY, 'augment': AUGMENT, 'imgsz': IMG_SIZE, 'seed': SEED,
    },
    'eval': {
        'real_images': {'conf_threshold': CONF_THRESH, 'iou_threshold': IOU_THRESH, 'imgsz': IMG_SIZE,
                         'real_data_root': REAL_DATA_ROOT,
                         'method': 'detection rate (quantitative) + manual qualitative review (see Section 12) '
                                    '-- no ground truth, no automated part-accuracy on real images (assumption #5)'},
        'crashcar_test': {'per_class_metrics_csv': per_class_csv,
                           'save_dir': os.path.join(EXP_DIR, 'crashcar_test_eval'),
                           'method': 'Ultralytics model.val(split=\'test\') -- synthetic-domain only, see notes.md'},
    },
    'date': date.today().isoformat(),
}
config_path = os.path.join(EXP_DIR, 'config.yaml')
with open(config_path, 'w') as f:
    yaml.dump(config, f, sort_keys=False)
print(f'✅ Saved → {config_path}')


In [ ]:
mapping_table_lines = ['| source class | source dataset | canonical class |', '|---|---|---|']
for k, v in CARPARTS_TO_CANONICAL.items():
    mapping_table_lines.append(f'| {k} | carparts-seg | {v if v else "(dropped)"} |')
for k, v in CRASHCAR_TO_CANONICAL.items():
    mapping_table_lines.append(f'| {k} | CrashCar101 | {v if v else "(dropped)"} |')
mapping_table_md = '\n'.join(mapping_table_lines)

detection_lines = '\n'.join(
    f"  - {r['subfolder']}/: {r['exp02_detection_rate_pct']}% -> {r['exp03_detection_rate_pct']}%  (delta {r['delta_pct']:+.2f} pts)"
    for r in detection_comparison_df.to_dict('records')
)

crashcar_test_block = (
    per_class_df.to_string(index=False) if not per_class_df.empty
    else '(per-class extraction failed -- see Section 11b output / test_metrics.results_dict)'
)
n_review_candidates = len(review_candidates) if 'review_candidates' in dir() else 0

notes_md = f"""# {EXP_ID} — CrashCar101 + cropped carparts-seg, targeting part classification
Date: {date.today().isoformat()}
Purpose (variables under test): Does adding CrashCar101 (synthetic part+damage segmentation)
to exp02's cropped-carparts training data improve PART CLASSIFICATION (not just detection) on
real CrashLens close-ups? NOTE: two variables changed vs exp02 simultaneously -- dataset
(+CrashCar101) AND training recipe (+early stopping, +weight decay, +freeze warmup). See
'Recommended next step' below for an ablation to attribute the effect if needed.
Dataset version: {DATASET_VERSION} (archived at {DRIVE_DATASETS_DIR})
  - CrashCar101: {CRASHCAR_HF_REPO}, subsampled frac={CRASHCAR_SAMPLE_FRAC} cap={CRASHCAR_MAX_IMAGES}, split={split_kind}
  - cropped-carparts: reused verbatim from exp02 ({EXP02_DATASET_DIR})
Class mapping (source -> canonical, full table -- audit this):
{mapping_table_md}
Config (lr, epochs, backbone frozen?, imgsz, augment): Ultralytics defaults + weight_decay={WEIGHT_DECAY},
  freeze={FREEZE_LAYERS} layers for {FREEZE_EPOCHS} warmup epochs then unfrozen for up to {FINETUNE_EPOCHS}
  epochs with patience={PATIENCE}, imgsz={IMG_SIZE}, seed={SEED}, augment={AUGMENT}
Training metrics (final epoch, from results.csv): {train_metrics_final if train_metrics_final else '(fill in after running)'}
Validation metrics (independent set): see results.csv/results.png -- NOTE val is CrashCar101(held-out)
  + cropped-carparts(val), both synthetic-leaning (see Weaknesses) -- NOT the real CrashLens domain.
Real CrashLens detection rate, before (exp02) -> after (exp03):
{detection_lines}
CrashCar101 held-out TEST split -- per-class precision/recall/mAP (SYNTHETIC DOMAIN ONLY, see
  Weaknesses -- this is NOT a real-CrashLens measurement, just an absolute quality bar for exp03):
{crashcar_test_block}
Real CrashLens PART CLASSIFICATION -- manual qualitative review ({n_review_candidates} candidate
  images pre-surfaced in Section 12, exp02_vs_exp03_detection.csv / manual_review_candidates.csv):
  (fill in after opening the candidate overlays + a spot-check of others -- e.g. "N of the M
  hood-misprediction candidates from exp02 now predict glass correctly in exp03" or "no change")
Visual predictions (paths): {os.path.join(EXP_DIR, 'val_preds')}/full/, .../cropped/
Confusion matrix (CrashCar101 test, synthetic domain): {os.path.join(EXP_DIR, 'crashcar_test_eval')}/confusion_matrix.png
Strengths: (fill in after reviewing val_preds/ overlays, the CrashCar101-test per-class table, and the
  confusion matrix -- did CrashCar101 fix the glass->hood-style errors specifically, or shift error
  mass elsewhere? Does the synthetic-test confusion matrix show the same glass<->hood confusion, or
  a different one?)
Weaknesses: (fill in -- also record explicitly: (1) per your decision (assumption #5), there is NO
  automated real-domain part-accuracy metric in this experiment at all -- the CrashCar101-test numbers
  above are a synthetic-domain-only proxy, and val is also entirely synthetic-leaning, so nothing in
  the automated pipeline confirms real-world transfer except the manual review; (2) CrashCar101
  taxonomy mapping (Section 4b) was filled in manually without independently verifying the paper's
  official table -- recheck if per-class results look inconsistent with what a class should be able to
  do; (3) mask-to-polygon conversion (Section 6) can merge adjacent same-class components into one
  instance -- spot-check labels, now used for train/val AND the test split)
Data/leakage checks done: exact-hash dedup across train/val/test (Section 8a); CrashCar101's own
  official test split used where available, so it was never seen during CrashCar101 pretraining/
  train/val either; carparts-seg splits kept disjoint at the source (train/val only, no test --
  cropped-carparts doesn't contribute to Section 11's metric). Real CrashLens images were never part
  of the training dataset object at all in this design.
  NOT done: full pairwise near-duplicate search within CrashCar101's ~101k images (see Section 8a
  rationale -- relies on the dataset's own split integrity instead).
Lessons learned: (fill in -- does more data + regularization actually fix WRONG-PART errors on real
  images (per the manual review), and does that track with what the CrashCar101-test numbers show, or
  did the model improve on synthetic data without the real-world problem actually changing?)
Recommended next step: (fill in) If the CrashCar101-test gain (or lack of one) needs to be attributed
  to the dataset vs. the recipe change, run an ablation: exp03b_crashcar_plus_carparts_noreg -- same
  unified dataset, but exp02's ORIGINAL recipe (no freeze warmup, no patience, default weight_decay,
  single-phase training) -- isolates the dataset effect. Also consider: if the manual review looks
  promising but you want more confidence than eyeballing gives, a small hand-labeled real GT set (even
  10-15 images) would let you actually measure real-domain part accuracy instead of inferring it from
  a synthetic proxy -- worth it once you have a reason to believe the answer, not before.
"""

notes_path = os.path.join(EXP_DIR, 'notes.md')
with open(notes_path, 'w', encoding='utf-8') as f:
    f.write(notes_md)
print(f'✅ Saved → {notes_path}')
print('⚠️  notes.md has placeholders -- fill in Strengths/Weaknesses/Lessons/Next step after reviewing '
      'val_preds/ overlays and the confusion matrix before calling this experiment finished.')


## ✅ Section 14 — Summary

In [ ]:
print('=' * 70)
print(f'STAGE 2 — {EXP_ID}')
print('=' * 70)
for r in detection_comparison_df.to_dict('records'):
    print(f"{r['subfolder']:8s} | detection: exp02 {r['exp02_detection_rate_pct']}% -> exp03 {r['exp03_detection_rate_pct']}% (delta {r['delta_pct']:+.2f} pts)")
if not per_class_df.empty:
    print(f"CrashCar101 test (synthetic domain) | mean precision {per_class_df['precision'].mean():.1%} "
          f"| mean recall {per_class_df['recall'].mean():.1%} | mean mAP50 {per_class_df['mAP50'].mean():.1%}")
else:
    print('CrashCar101 test metrics | extraction failed -- see test_metrics.results_dict (Section 11)')
print(f'Real CrashLens part classification | MANUAL REVIEW REQUIRED -- {n_review_candidates} candidate '
      'images pre-surfaced in Section 12, not yet judged')
print('=' * 70)
print(f'Artifacts saved to: {EXP_DIR}')
print('  - weights/best.pt, weights/last.pt')
print('  - results.csv, results.png (Ultralytics training curves, phase 2)')
print('  - real_eval_results.csv, real_eval_summary.csv, exp02_vs_exp03_detection.csv')
print('  - crashcar_test_eval/ (Ultralytics val output incl. confusion_matrix.png), crashcar_test_per_class_metrics.csv')
print('  - manual_review_candidates.csv, val_preds/full/, val_preds/cropped/ (annotated overlays)')
print('  - config.yaml, notes.md')
print()
print('Next: open manual_review_candidates.csv + val_preds/ overlays and judge the real-image part')
print('classification by eye (Section 12), review the CrashCar101-test confusion matrix (Section 11),')
print('fill in notes.md, then decide: accept exp03, run the exp03b no-reg ablation, or iterate on '
      'CRASHCAR_TO_CANONICAL.')
